# Create samplesheet

A simple notebook to create the samplesheet files to be used in the analyses:

In [ ]:
import pandas as pd

from utils import find_project_root

`find_project_root` is an utility function to detect the absolute path of the current 
directory: I use it (like the *R* `here` package) to avoid to use absolute paths in
the code. Determine where the data files are:

In [ ]:
data_dir = find_project_root() / 'data'

Read the dataframe coming from the sequencing run:

In [ ]:
# Read the dataframe from the file
df = pd.read_excel(data_dir / "MISEQ-MENIN.xlsx")

# Drop the columns that are not needed
df.drop(columns=["Unnamed: 3", "Unnamed: 4"], inplace=True)

# Rename the columns
df.rename(columns={
        "MISEQ*": "miseq",
        "PROGRESSIVO CAMPIONE": "sample_number",
        "TARGET": "target"},
    inplace=True)

# Display the dataframe
df.head()

Replace the `miseq` values with a progressive number:

In [ ]:
df['miseq'] = range(1, len(df) + 1)
df.head()

Try to determine the name programmatically using sample `sampleID`:

In [ ]:
df['forwardReads'] = df.apply(lambda row: f"data/240926_M04028_0172_000000000-LN722/{row['sample_number']}_S{row['miseq']}_L001_R1_001.fastq.gz", axis=1)
df['reverseReads'] = df.apply(lambda row: f"data/240926_M04028_0172_000000000-LN722/{row['sample_number']}_S{row['miseq']}_L001_R2_001.fastq.gz", axis=1)
df.head()

Open *metadata* file made using `scripts/metadata.ipynb`: we need to rename the sample
id relying on those files:

In [ ]:
metadata_df = pd.read_excel(data_dir / "metadata.xlsx")
metadata_df.head()

Join the two dataframes relying on sample number:

In [ ]:
merged_df = pd.merge(df, metadata_df, on='sample_number')
merged_df.head()

Now select all *fungi* samples and write into a `.csv` file:

In [ ]:
df_fungi = merged_df[merged_df['target_x'] == 'FUNGHI'][['sampleID', 'forwardReads', 'reverseReads']]
df_fungi.head()

In [ ]:
df_fungi.to_csv(data_dir / 'samplesheet_fungi.csv', index=False)

Do the same for the *bacteria* samples:

In [ ]:
df_bacteria = merged_df[merged_df['target_x'] == 'BATTERI'][['sampleID', 'forwardReads', 'reverseReads']]
df_bacteria.head()

In [ ]:
df_bacteria.to_csv(data_dir / 'samplesheet_bacteria.csv', index=False)